Dans ce TP nous allons voir le problème de comptage de mots.
D'abord nous verrons comment générer le fichier que nous allons compter puis nous verrons 3 méthodes sur ce fichier :
- une manière séquentiel en Python
- une manière inspirée de MapReduce toujours en python
- puis en utilisant le MapReduce de Hadoop

# Création de la liste de mots

In [32]:
import random
import time
from collections import defaultdict 

In [33]:
liste = ['GRIMACES\n','PULVERISER\n',
'HANTANT\n',
'VILLAGE\n',
'DRESSENT\n',
'DEFERES\n',
'CAPTEES\n',
'PEDANTE\n',
'LIVINGROOM\n',
'BATEAU\n',
'RETIREE\n',
'DISPOSAIT\n',
'SIERA\n',
'ARRETEZ\n',
'REPRENDRONS\n',
'RENAISSANTE\n',
'PAIES\n',
'MONTREALAIS\n',
'PRANGINS\n',
'JARDINER\n',]

In [34]:
with open("mots.txt","w",encoding="utf-8") as f :
    for i in range(1000):
        f.write(random.choice(liste))
fichier = "mots.txt"

# Méthode séquentielle

In [47]:
def compter(fichier):
    with open(fichier,"r",encoding="utf-8") as f :
        mots = f.read().split()
    compteur = {}
    for mot in mots :
        compteur[mot] = compteur.get(mot,0) + 1
    return compteur
start_time = time.perf_counter()
resultat = compter(fichier)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Temps d'exécution : {elapsed_time:.6f} secondes")
for mot , nb in sorted (resultat.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : {nb}")

Temps d'exécution : 0.000822 secondes
DRESSENT : 62
BATEAU : 54
SIERA : 53
DEFERES : 53
RENAISSANTE : 52
RETIREE : 52
CAPTEES : 52
MONTREALAIS : 52
REPRENDRONS : 51
PAIES : 51
DISPOSAIT : 51
PRANGINS : 49
VILLAGE : 49
LIVINGROOM : 48
HANTANT : 48
JARDINER : 48
PULVERISER : 45
GRIMACES : 44
PEDANTE : 44
ARRETEZ : 42


On voit bien que cette méthode est extrèmement efficace pour des fichiers d'une centaine de mots voir même d'un millier de mots

Maintenant nous allons passer a la partie suivante ou nous allons répliquer le fonctionnement de MapReduce en Python

# MapReduce en Python

D'abord nous allons mapper donc la nous créons une liste de tuple avec chaque mots de notre fichier associé avec la valeur 1

In [48]:
def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

Ensuite on shuffle. Cela ne reviens pas a tout mélanger mais nous allons regrouper les valeur associés a un même mot

In [49]:
def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

Pour finir on reduce, c'est a dire qu'on ajoute chaque valeur associé a chaque mots.

In [50]:
def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced

Une éxécution ressemblerai ainsi a cela :

In [51]:
mapped = []
start_time = time.perf_counter()
with open (fichier , "r", encoding="utf -8") as f:
    for ligne in f:
        mapped.extend(mapper(ligne))
grouped = shuffle(mapped)
reduced = reducer(grouped)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Temps d'exécution : {elapsed_time:.6f} secondes")
for mot , count in sorted (reduced.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : { count }")

Temps d'exécution : 0.001733 secondes
DRESSENT : 62
BATEAU : 54
SIERA : 53
DEFERES : 53
RENAISSANTE : 52
RETIREE : 52
CAPTEES : 52
MONTREALAIS : 52
REPRENDRONS : 51
PAIES : 51
DISPOSAIT : 51
PRANGINS : 49
VILLAGE : 49
LIVINGROOM : 48
HANTANT : 48
JARDINER : 48
PULVERISER : 45
GRIMACES : 44
PEDANTE : 44
ARRETEZ : 42


On voit bien que le compte est exactement le même et que l'ordre de grandeur au niveau du temps d'exécution est aussi le même. On va maintenant passer au MapReduce de Hadoop

# Hadoop MapReduce

Pour la partie de Hadoop nous devrons d'abord avoir une liste bien plus grande de mots car Hadoop n'est pas très performant pour les petits fichiers mais brillent pour les fichier plus volumineux.

In [54]:
with open(fichier,"r",encoding="utf-8") as source:
    contenu = source.read()
fichier_long = "mots_long.txt"
with open(fichier_long,"w",encoding="utf-8") as sortie:
    for _ in range(10000):
        sortie.write(contenu)
        sortie.write("\n")

On va d'abord voir le temps d'exécution avec les méthodes en python :

In [55]:
start_time = time.perf_counter()
resultat_long = compter("mots_long.txt")
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Temps d'exécution : {elapsed_time:.6f} secondes")
for mot , nb in sorted (resultat_long.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : {nb}")

Temps d'exécution : 3.635563 secondes
DRESSENT : 620000
BATEAU : 540000
SIERA : 530000
DEFERES : 530000
RENAISSANTE : 520000
RETIREE : 520000
CAPTEES : 520000
MONTREALAIS : 520000
REPRENDRONS : 510000
PAIES : 510000
DISPOSAIT : 510000
PRANGINS : 490000
VILLAGE : 490000
LIVINGROOM : 480000
HANTANT : 480000
JARDINER : 480000
PULVERISER : 450000
GRIMACES : 440000
PEDANTE : 440000
ARRETEZ : 420000


Quand le fichier était seulement 100 fois plus grand cela ne changeait pas grand chose pour Python mais Hadoop mettait 3.866s de calcul maintenant quand le fichier est 10000 fois plus grand Python mets 3.636s et Hadoop met 4,401s. On voit bien que plus la taille du fichier augmente plus cela est long pour Python mais pour Hadoop cela n'augmente pas aussi rapidement.